## Importing data:

In [ ]:
import pandas as pd

train_path = '/kaggle/input/competitions/nlp-getting-started/train.csv'
test_path = '/kaggle/input/competitions/nlp-getting-started/test.csv'

train_df = pd.read_csv(train_path)
print('Training set: ',train_df.shape)
display(train_df.head(3))

test_df = pd.read_csv(test_path)
print('Test set: ',test_df.shape)
display(test_df.head(3))

In [ ]:
print(f"Total samples in the training set are: {len(train_df)}")

In [ ]:
target = 'target'

#we'll be using bag-of-word model so we will only require text column and the target
X = train_df['text'].values
y = train_df[target].values

## Preparing data:

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y, 
                                               test_size=0.2,
                                               stratify=y, 
                                               random_state=42)

#converting the arrays from train_test_split to list of stringas, cuz AutoTokenizer doesnt accept numpy arrays or pandas series
X_train = X_train.astype(str).tolist()
X_val = X_val.astype(str).tolist()

We'll tokenize our data using Hugging Face Transformers tokenizer "**AutoTokenizer**"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_enc = tokenizer(
    X_train,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

val_enc = tokenizer(
    X_val,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
#Creating label tensors
import torch

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [ ]:
#Creating dataset objects
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    train_enc['input_ids'],
    train_enc['attention_mask'],
    y_train
)

val_dataset = TensorDataset(
    val_enc['input_ids'],
    val_enc['attention_mask'],
    y_val
)

#Loader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

## Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
#enabling GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### Training loop:

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train() #switching to train mode

for epoch in range(1):
    print(f"\nEpoch {epoch+1}")

    # 🔹 TRAINING
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Train Loss:", loss.item())

    #VALIDATION
    model.eval()  #switching to evaluate mode
    
    total_val_loss = 0

    with torch.no_grad():   
        for batch in val_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print("Val Loss:", avg_val_loss)

    model.train()  # switch back to training mode

## Making predictions on test data:

In [ ]:
#Preparing test data
test_enc = tokenizer(
    test_df['text'].astype(str).to_list(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
test_dataset = TensorDataset(
    test_enc['input_ids'],
    test_enc['attention_mask']
)

test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
model.eval()

predictions = []

with torch.no_grad():  # no gradients needed
    for batch in test_loader:
        input_ids, attention_mask = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)  # convert to 0/1

        predictions.extend(preds.cpu().numpy())

In [ ]:
print(predictions[:10])

## Submission

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: predictions
})
submission.to_csv('submission.csv', index=False)